In [1]:
!pip install ultralytics -q

import os
import yaml
import json
from ultralytics import YOLO
import torch
import pandas as pd

# ==========================================
# 1. KAGGLE PATHS (From your screenshot & logs)
# ==========================================
BASE_DIR = '/kaggle/input/datasets/prosenjitmondol/complete-vindr-spinexr'

IMG_TRAIN = f'{BASE_DIR}/vindr-spinexr-a-large-annotated-medical-image-dataset/vindr-spinexr-a-large-annotated-medical-image-dataset/train_png'
IMG_VAL = f'{BASE_DIR}/vindr-spinexr-a-large-annotated-medical-image-dataset/vindr-spinexr-a-large-annotated-medical-image-dataset/test_png'

COCO_TRAIN = f'{BASE_DIR}/coco format/train_coco.json'
COCO_VAL = f'{BASE_DIR}/coco format/test_coco.json'

# ==========================================
# 2. CREATE YOLO DIRECTORY STRUCTURE
# ==========================================
WORK_BASE = '/kaggle/working/vindr_yolo'
os.makedirs(f'{WORK_BASE}/images/train', exist_ok=True)
os.makedirs(f'{WORK_BASE}/images/val', exist_ok=True)
os.makedirs(f'{WORK_BASE}/labels/train', exist_ok=True)
os.makedirs(f'{WORK_BASE}/labels/val', exist_ok=True)

print("Setting up YOLO dataset and creating .txt labels...")

# Symlink images to the working directory so YOLO can find them
for img in os.listdir(IMG_TRAIN):
    if img.endswith(('.png', '.jpg')):
        dst = os.path.join(f'{WORK_BASE}/images/train', img)
        if not os.path.exists(dst): os.symlink(os.path.join(IMG_TRAIN, img), dst)

for img in os.listdir(IMG_VAL):
    if img.endswith(('.png', '.jpg')):
        dst = os.path.join(f'{WORK_BASE}/images/val', img)
        if not os.path.exists(dst): os.symlink(os.path.join(IMG_VAL, img), dst)

# Function to convert COCO JSON to YOLO TXT
def convert_coco_to_yolo(json_path, labels_dir):
    with open(json_path) as f:
        data = json.load(f)

    cat_map = {cat['id']: i for i, cat in enumerate(data['categories'])}
    img_map = {img['id']: img for img in data['images']}

    for ann in data['annotations']:
        img = img_map[ann['image_id']]
        x_min, y_min, w, h = ann['bbox']

        # YOLO formatting (x_center, y_center, width, height)
        x_center = (x_min + w / 2) / img['width']
        y_center = (y_min + h / 2) / img['height']
        w_norm = w / img['width']
        h_norm = h / img['height']
        cls_id = cat_map[ann['category_id']]

        txt_filename = img['file_name'].rsplit('.', 1)[0] + '.txt'
        with open(os.path.join(labels_dir, txt_filename), 'a') as out_f:
            out_f.write(f"{cls_id} {x_center} {y_center} {w_norm} {h_norm}\n")

    return {i: cat['name'] for i, cat in enumerate(data['categories'])}

print("Converting Train Labels...")
class_names_dict = convert_coco_to_yolo(COCO_TRAIN, f'{WORK_BASE}/labels/train')
print("Converting Val Labels...")
convert_coco_to_yolo(COCO_VAL, f'{WORK_BASE}/labels/val')

# ==========================================
# 3. GENERATE YOLO YAML
# ==========================================
yaml_content = {
    'path': WORK_BASE,
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names_dict),
    'names': class_names_dict
}

YAML_PATH = '/kaggle/working/vindr_data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✓ Dataset setup complete! YAML created at {YAML_PATH}")

# ==========================================
# 4. ABLATION SELECTOR
# ==========================================
# Options: "NO_COPY_PASTE" or "NO_FOCAL_LOSS"
ABLATION_MODE = "NO_COPY_PASTE" 

EPOCHS = 55
BATCH_SIZE = 12
IMG_SIZE = 640

print("\n" + "="*70)
print(f"STARTING ABLATION EXPERIMENT: {ABLATION_MODE}")
print("="*70)

# Configure Hyperparameters
train_args = {
    'data': YAML_PATH,
    'epochs': EPOCHS,
    'batch': BATCH_SIZE,
    'imgsz': IMG_SIZE,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'optimizer': 'AdamW',
    'lr0': 0.0001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'copy_paste': 0.2, 
    'mosaic': 1.0,
    'amp': True,
    'project': '/kaggle/working/ablation_runs',
    'seed': 42
}

# Apply the handicap to the training run
if ABLATION_MODE == "NO_COPY_PASTE":
    train_args['copy_paste'] = 0.0
    train_args['name'] = 'ablation_no_copypaste'
    print("-> Copy-Paste Augmentation disabled (copy_paste=0.0)")
    
elif ABLATION_MODE == "NO_FOCAL_LOSS":
    train_args['name'] = 'ablation_no_focalloss'
    print("-> Focal Loss disabled (Reverting to standard BCE Loss)")

# Initialize and Train
model = YOLO('yolo11l.pt')
print(f"\nTraining {train_args['name']} for {EPOCHS} epochs...")
results = model.train(**train_args)

# Extract and print Results
run_dir = os.path.join(train_args['project'], train_args['name'])
results_csv = os.path.join(run_dir, 'results.csv')

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip() 
    best_map50 = df['metrics/mAP50(B)'].max()
    
    print("\n" + "="*70)
    print(f"ABLATION RESULT FOR: {ABLATION_MODE}")
    print("="*70)
    print(f"Final mAP@0.5: {best_map50:.4f} ({best_map50*100:.2f}%)")
    
    baseline_map50 = 40.04  # Your paper's baseline
    ablation_map50 = best_map50 * 100
    drop = ablation_map50 - baseline_map50
    print(f"Baseline mAP@0.5: {baseline_map50}%")
    print(f"Performance Drop (ΔmAP): {drop:.2f}%")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23